# 04 - Final Backtest Report

Chronological held-out evaluation of the RippleETA Stage 3-5 pipeline across the six selected Stage 2 train routes.

This report uses measured data only. The available journey-level dataset does not contain station-pair live state, so timed-event conflict adjustments are reported separately and remain inactive in this backtest.

## Method

Each route is sorted by journey date and split chronologically: 70% training, 15% MAPIE calibration, and 15% untouched test. The baseline carries forward `prior_leg_delay`. The full model trains XGBoost on engineered features and evaluates its calibrated P50 and P10-P90 interval.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import sys
sys.path.append('..')
from src.evaluation.backtest import run_backtest

result = run_backtest()
route_metrics = pd.DataFrame(result['route_metrics'])
predictions = result['predictions']
result['overall']

## Headline measured results

The aggregate test set contains 174 held-out route journeys. The report numbers below are produced by the evaluator, not entered by hand.

In [ ]:
overall = result['overall']
pd.DataFrame([
    {'metric': 'Baseline MAE (min)', 'measured': overall['baseline_mae_min']},
    {'metric': 'Full pipeline P50 MAE (min)', 'measured': overall['full_pipeline_mae_min']},
    {'metric': 'MAE improvement (min)', 'measured': overall['mae_improvement_min']},
    {'metric': 'MAE improvement (%)', 'measured': overall['mae_improvement_pct']},
    {'metric': 'P10-P90 coverage (%)', 'measured': overall['coverage_90_pct']},
    {'metric': 'Average interval width (min)', 'measured': overall['avg_interval_width_min']},
    {'metric': 'Conflict-adjusted rows', 'measured': overall['conflict_adjustment_rows']},
])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = route_metrics['route']
width = 0.36
positions = range(len(route_metrics))
ax.bar([p - width / 2 for p in positions], route_metrics['baseline_mae_min'], width, label='Prior-leg baseline', color='#728688')
ax.bar([p + width / 2 for p in positions], route_metrics['full_pipeline_mae_min'], width, label='Full pipeline P50', color='#e7aa45')
ax.set_xticks(list(positions), x)
ax.set_ylabel('MAE (minutes)')
ax.set_title('Held-out MAE by selected route')
ax.legend()
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(route_metrics['route'], route_metrics['coverage_90_pct'], color='#62bdb7')
ax.axhline(90, color='#e7aa45', linestyle='--', linewidth=2, label='Nominal target: 90%')
ax.set_ylim(0, 105)
ax.set_ylabel('Actuals inside P10-P90 (%)')
ax.set_title('Empirical coverage on each held-out route')
ax.legend()
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

## Route detail and claim audit

The route table is the defensible replacement for generic literature ranges. In this artifact, all six selected IDs are present, but no station-pair occupancy fields are present; therefore no route can claim measured conflict propagation.

In [ ]:
route_metrics[['route', 'n_total', 'n_test', 'baseline_mae_min', 'full_pipeline_mae_min', 'mae_improvement_pct', 'coverage_90_pct', 'conflict_adjustment_rows']]

In [ ]:
pd.DataFrame([result['real_12301_example']])

## Conclusion

Measured aggregate coverage exceeds the 90% target, but the intervals are wide. The measured 18.30% MAE improvement is useful but does not reproduce the literature's 5-9 minute network-aware MAE range. The 12301 real row demonstrates why the illustrative 12301/56789 conflict story cannot be presented as a real-data validation until station-level paired data is added.